In [1]:
!pip install torch torchvision torchaudio gymnasium gymnasium[classic-control]

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from collections import deque

import matplotlib.pyplot as plt
from IPython.display import Video
import os

In [3]:
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


In [4]:
# CONFIGURATION
EPISODES        = 1000
GAMMA           = 0.99
LR              = 1e-3
BATCH_SIZE      = 64
BUFFER_SIZE     = 10_000
EPSILON_START   = 1.0
EPSILON_END     = 0.01
EPSILON_DECAY   = 0.995
TARGET_UPDATE   = 10
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
# Neural Network for DQN

class QNetwork(nn.Module):
  def __init__(self, state_dim: int, action_dim: int):
    super(QNetwork, self).__init__()

    self.embed = nn.Embedding(state_dim, 64)

    # NN Layer: can be change in future
    self.net = nn.Sequential(
        nn.Linear(64, 128),
        nn.ReLU(),
        nn.Linear(128, 128),
        nn.ReLU(),
        nn.Linear(128, action_dim)
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    x = x.long()
    x = self.embed(x)
    return self.net(x)

In [6]:
# Replay Buffer

class ReplayBuffer:
  def __init__(self, buffer_size: int):
    self.buffer = deque(maxlen=buffer_size)

  def push(self, state, action, reward, next_state, done):
    self.buffer.append((state, action, reward, next_state, done))

  def sample(self, batch_size: int):
    batch = random.sample(self.buffer, batch_size)
    states, action, reward, next_state, done = zip(*batch)

    return (
        torch.LongTensor(np.array(states)).to(DEVICE),
        torch.LongTensor(action).to(DEVICE),
        torch.FloatTensor(reward).to(DEVICE),
        torch.LongTensor(np.array(next_state)).to(DEVICE),
        torch.FloatTensor(done).to(DEVICE)
    )

  def __len__(self):
    return len(self.buffer)

In [7]:
# DQN Agent
class DQNAgent:
  def __init__(self, state_dim: int, action_dim: int):
    self.action_dim = action_dim

    self.online_net = QNetwork(state_dim, action_dim).to(DEVICE)
    self.target_net = QNetwork(state_dim, action_dim).to(DEVICE)
    self.target_net.load_state_dict(self.online_net.state_dict())
    self.target_net.eval()

    self.optimizer = optim.Adam(self.online_net.parameters(), lr=LR)
    self.loss_fn = nn.SmoothL1Loss()

    self.buffer = ReplayBuffer(BUFFER_SIZE)

    self.epsilon = EPSILON_START

  def select_action(self, state, action_mask) -> int:
    if random.random() < self.epsilon:
      # only for valid action
      actions = [i for i in range(len(action_mask)) if action_mask[i] == 1]
      return random.choice(actions)

    with torch.no_grad():
      state_t = torch.LongTensor(np.array([state])).to(DEVICE)
      q_values = self.online_net(state_t)
      return q_values.argmax(dim=1).item()

  def learn(self):
    if len(self.buffer) < BATCH_SIZE:
      return

    states, actions, rewards, next_states, dones = self.buffer.sample(BATCH_SIZE)
    q_pred = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
      q_next = self.target_net(next_states).max(dim=1).values
      q_target = rewards + GAMMA * q_next * (1.0 - dones)

    loss = self.loss_fn(q_pred, q_target)
    self.optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(self.online_net.parameters(), max_norm=10.0)
    self.optimizer.step()

  def update_target_net(self):
    self.target_net.load_state_dict(self.online_net.state_dict())

  def decay_epsilon(self):
    self.epsilon = max(EPSILON_END, self.epsilon * EPSILON_DECAY)

In [8]:
# Reward shapping with potential func
PASSANGER_IN_TAXI = 4
PASSANGER_POSITION = {
    0: (0, 0),
    1: (0, 4),
    2: (4, 0),
    3: (4, 3),
}
DESTINATION_POSITION = {
    0: (0, 0),
    1: (0, 4),
    2: (4, 0),
    3: (4, 3),
}

def manhattan_distance(p1, p2):
  return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

# More closer to dest or pass, more reward from potential func
def potential(env, state):
  taxi_row, taxi_col, pass_idx, dest_idx = env.unwrapped.decode(state)
  taxi_pos = (taxi_row, taxi_col)

  # if passager not in taxi, potensial focus on distance to passager
  if pass_idx != PASSANGER_IN_TAXI:
    pass_poss = PASSANGER_POSITION[pass_idx]
    return -manhattan_distance(taxi_pos, pass_poss)

  # passager in taxi, focus to destination
  dest_pos = DESTINATION_POSITION[dest_idx]
  return -manhattan_distance(taxi_pos, dest_pos)

def reward_shaping(env, state, next_state, raw_reward):
  return raw_reward + GAMMA * potential(env, next_state) - potential(env, state)


In [9]:
# train function

def train():
  env = gym.make("Taxi-v4")
  state_dim = env.observation_space.n
  action_dim = env.action_space.n

  agent = DQNAgent(state_dim, action_dim)
  reward_history = []

  print(f"Training DQN on Taxi-v4 | Device: {DEVICE}")
  print(f"State dim: {state_dim}, Action dim: {action_dim}")
  print("-" * 60)

  for episode in range (1, EPISODES + 1):
    state, info = env.reset()
    action_mask = info['action_mask']

    total_reward = 0
    done = False

    while not done:
      action = agent.select_action(state, action_mask)
      next_state, reward, terminated, truncated, info = env.step(action)
      done = terminated or truncated

      r_store = reward_shaping(env, state, next_state, reward)
      agent.buffer.push(state, action, r_store, next_state, float(done))
      state = next_state
      action_mask = info['action_mask']
      total_reward += reward

      agent.learn()

    agent.decay_epsilon()
    reward_history.append(total_reward)

    # LOGGING
    if episode % 10 == 0:
      avg_reward = np.mean(reward_history[-10:])
      print(
          f"Episode {episode:4d} | "
          f"Avg Reward (last 10): {avg_reward:6.1f} | "
          f"Epsilon: {agent.epsilon:.3f}"
      )

    if episode % TARGET_UPDATE == 0:
      agent.update_target_net()

  env.close()
  return agent, reward_history

In [10]:
agent, history = train()

print("\nTraining selesai!")
print(f"Total episode: {len(history)}")
print(f"Reward akhir (rata-rata semua episode): {np.mean(history):.1f}")

Training DQN on Taxi-v4 | Device: cuda
State dim: 500, Action dim: 6
------------------------------------------------------------
Episode   10 | Avg Reward (last 10): -209.0 | Epsilon: 0.951
Episode   20 | Avg Reward (last 10): -203.6 | Epsilon: 0.905
Episode   30 | Avg Reward (last 10): -191.4 | Epsilon: 0.860
Episode   40 | Avg Reward (last 10): -193.8 | Epsilon: 0.818
Episode   50 | Avg Reward (last 10): -171.8 | Epsilon: 0.778
Episode   60 | Avg Reward (last 10): -188.1 | Epsilon: 0.740
Episode   70 | Avg Reward (last 10): -195.5 | Epsilon: 0.704
Episode   80 | Avg Reward (last 10): -128.6 | Epsilon: 0.670
Episode   90 | Avg Reward (last 10): -162.6 | Epsilon: 0.637
Episode  100 | Avg Reward (last 10): -113.3 | Epsilon: 0.606
Episode  110 | Avg Reward (last 10): -129.9 | Epsilon: 0.576
Episode  120 | Avg Reward (last 10): -184.9 | Epsilon: 0.548
Episode  130 | Avg Reward (last 10): -136.8 | Epsilon: 0.521
Episode  140 | Avg Reward (last 10): -115.5 | Epsilon: 0.496
Episode  150 | A

In [13]:
def evaluate(agent: DQNAgent, n_episodes: int = 10):
    env = gym.make("Taxi-v4")
    old_epsilon = agent.epsilon
    agent.epsilon = 0.0

    rewards = []
    for _ in range(n_episodes):
        state, info = env.reset()
        action_mask = info['action_mask']
        total_reward = 0
        done = False
        while not done:
            action = agent.select_action(state, action_mask)
            state, reward, terminated, truncated, info = env.step(action)
            action_mask = info['action_mask']
            done = terminated or truncated

            total_reward += reward

        rewards.append(total_reward)

    agent.epsilon = old_epsilon
    env.close()

    print(f"\nEvaluasi ({n_episodes} episode):")
    print(f"  Rata-rata reward : {np.mean(rewards):.1f}")
    print(f"  Reward tertinggi : {np.max(rewards):.1f}")
    print(f"  Reward terendah  : {np.min(rewards):.1f}")
    return rewards

evaluate(agent, n_episodes=20)


Evaluasi (20 episode):
  Rata-rata reward : 8.0
  Reward tertinggi : 15.0
  Reward terendah  : 3.0


[10, 8, 11, 11, 5, 6, 9, 10, 4, 12, 11, 15, 3, 5, 7, 9, 4, 9, 6, 5]